
# Week 5: Evaluation and Backtesting


**Owner:** Fatima Nasir  
**Role:** Evaluation / Backtesting Lead  
**Deliverable:** Evaluation Plan and Metrics Code


## Objective
Evaluate how well our market stress early-warning system identifies periods of market stress and whether our combined signals provide useful information beyond VIX alone.

## Evaluation Metrics
- **Precision:** Measures how many predicted stress periods are actual stress periods.
- **Recall:** Measures how many actual stress periods are detected.
- **F1-score:** Balances precision and recall.
- **PR-AUC:** Evaluates performance when stress periods are rare.
- **ROC-AUC:** Measures the model's ability to distinguish stress from non-stress periods.
- **False Alarm Rate:** Measures how often the model incorrectly signals stress.

## Backtesting Plan
1. Use historical stress labels as the target variable.
2. Split the data chronologically to avoid future information leakage.
3. Train a baseline model using VIX only.
4. Train a comparison model using VIX and additional market stress indicators.
5. Compare both models using the same evaluation metrics and test dates.
6. Evaluate FinBERT sentiment features separately for 2022, where sentiment data is available.

## Expected Outcome
Determine whether the additional market indicators improve stress detection compared with VIX alone, while reporting false alarms and limitations.



## 1. Setup

Import the required libraries for data processing, analysis, and model evaluation.


In [19]:

# Step 1: Import libraries for evaluation

import pandas as pd
import numpy as np

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix
)

print("Evaluation libraries loaded successfully!")


Evaluation libraries loaded successfully!



## 2. Stress Labels and Data Preparation

Historical market stress labels are used to identify stress and non-stress periods. The data is checked for missing values and class distribution before model evaluation.


In [20]:

# Step 2: Find available stress label files

from pathlib import Path

ROOT = Path.cwd().parent
processed_dir = ROOT / "data" / "processed"

for file in sorted(processed_dir.rglob("*")):
    if file.is_file() and "stress" in file.name.lower():
        print(file.relative_to(ROOT))


data\processed\stress_labels_daily.parquet
data\processed\stress_periods.csv


In [21]:

# Step 3: Load daily stress labels

stress_df = pd.read_parquet(
    ROOT / "data" / "processed" / "stress_labels_daily.parquet"
)

print("Dataset shape:", stress_df.shape)
print("Columns:", stress_df.columns.tolist())
print("Date range:", stress_df.index.min(), "to", stress_df.index.max())

display(stress_df.head())


Dataset shape: (5490, 4)
Columns: ['vol_centered', 'stress_day_raw', 'episode_id', 'stress']
Date range: 2005-01-03 00:00:00 to 2026-10-28 00:00:00


,vol_centered,stress_day_raw,episode_id,stress
Date,,,,
2005-01-03,NaN,NaN,NaN,NaN
2005-01-04,NaN,NaN,NaN,NaN
2005-01-05,NaN,NaN,NaN,NaN
2005-01-06,NaN,NaN,NaN,NaN
2005-01-07,NaN,NaN,NaN,NaN


In [22]:

# Step 4: Check stress label distribution

print("Missing stress labels:", stress_df["stress"].isna().sum())

print("\nStress label counts:")
print(stress_df["stress"].value_counts(dropna=False))

print("\nAvailable labeled dates:")
valid_stress = stress_df.dropna(subset=["stress"]).copy()

print("First:", valid_stress.index.min())
print("Last:", valid_stress.index.max())
print("Total:", len(valid_stress))


Missing stress labels: 21

Stress label counts:
stress
0.0    4839
1.0     630
NaN      21
Name: count, dtype: int64

Available labeled dates:
First: 2005-01-19 00:00:00
Last: 2026-10-14 00:00:00
Total: 5469



## 3. Evaluation Metrics

Model performance is measured using accuracy, precision, recall, F1-score, ROC-AUC, PR-AUC, and false alarm rate.

Since stress events are less frequent than normal market conditions, recall, PR-AUC, and false alarm rate are especially useful for evaluating stress detection performance.


In [23]:

# Step 5: Define evaluation metrics

def evaluate_model(y_true, y_pred, y_prob):
    results = {
        "Accuracy": accuracy_score(y_true, y_pred),
        "Precision": precision_score(y_true, y_pred, zero_division=0),
        "Recall": recall_score(y_true, y_pred, zero_division=0),
        "F1": f1_score(y_true, y_pred, zero_division=0),
        "ROC-AUC": roc_auc_score(y_true, y_prob),
        "PR-AUC": average_precision_score(y_true, y_prob)
    }

    tn, fp, fn, tp = confusion_matrix(
        y_true, y_pred, labels=[0, 1]
    ).ravel()

    results["False Alarm Rate"] = (
        fp / (fp + tn) if (fp + tn) > 0 else np.nan
    )

    return pd.Series(results)
    
print("Evaluation function ready!")


Evaluation function ready!



## 4. VIX Baseline and Model Comparison

A VIX-only baseline model is compared with a combined model using VIX and additional market indicators.

Both models are evaluated using the same chronological training and testing periods to ensure a fair comparison and reduce the risk of data leakage.

The goal is to determine whether additional indicators improve market stress detection beyond VIX alone.


In [24]:

# Step 6: Inspect available market indicators

market = pd.read_parquet(
    ROOT / "data" / "processed" / "daily_panel.parquet"
)

print("Market data shape:", market.shape)
print("\nAvailable columns:")
print(market.columns.tolist())

print("\nDate range:")
print(market.index.min(), "to", market.index.max())

display(market.head(3))


Market data shape: (6997, 28)

Available columns:
['SPY', 'XLB', 'XLE', 'XLF', 'XLI', 'XLK', 'XLP', 'XLU', 'XLV', 'XLY', 'SPY_ret', 'XLB_ret', 'XLE_ret', 'XLF_ret', 'XLI_ret', 'XLK_ret', 'XLP_ret', 'XLU_ret', 'XLV_ret', 'XLY_ret', 'STLFSI4', 'NFCI', 'VIXCLS', 'BAMLH0A0HYM2', 'T10Y2Y', 'stress', 'avg_corr_60d', 'absorption_250d']

Date range:
1999-01-05 00:00:00 to 2026-10-28 00:00:00


,SPY,XLB,XLE,XLF,XLI,XLK,XLP,XLU,XLV,XLY,...,XLV_ret,XLY_ret,STLFSI4,NFCI,VIXCLS,BAMLH0A0HYM2,T10Y2Y,stress,avg_corr_60d,absorption_250d
date,,,,,,,,,,,,,,,,,,,,,
1999-01-05,76.952858,6.040749,5.646690,11.248712,15.244627,12.562191,14.132486,5.807241,17.525572,9.684027,...,0.015541,0.016707,0.3872,-0.201,24.46,NaN,0.10,0,NaN,NaN
1999-01-06,78.808105,6.147099,5.833012,11.605337,15.572269,12.933377,14.303343,5.867515,17.816624,9.959071,...,0.016471,0.028006,0.3872,-0.201,23.34,NaN,0.11,0,NaN,NaN
1999-01-07,78.421593,6.100304,5.806396,11.783652,15.437354,12.892775,14.140624,5.846416,17.733469,9.947616,...,-0.004678,-0.001151,0.3872,-0.201,24.37,NaN,0.14,0,NaN,NaN


In [25]:

# Step 7: Prepare data for model comparison

features = [
    "VIXCLS",
    "STLFSI4",
    "NFCI",
    "SPY_ret",
    "avg_corr_60d",
    "absorption_250d"
]

# Join market indicators with stress labels
model_df = market[features].join(
    stress_df[["stress"]],
    how="inner"
)

# Remove rows with missing values
model_df = model_df.dropna().sort_index()

print("Model dataset shape:", model_df.shape)
print("\nStress label counts:")
print(model_df["stress"].value_counts())

print("\nDate range:")
print(model_df.index.min(), "to", model_df.index.max())

display(model_df.head())


Model dataset shape: (5469, 7)

Stress label counts:
stress
0.0    4839
1.0     630
Name: count, dtype: int64

Date range:
2005-01-19 00:00:00 to 2026-10-14 00:00:00


,VIXCLS,STLFSI4,NFCI,SPY_ret,avg_corr_60d,absorption_250d,stress
2005-01-19,13.18,-0.6946,-0.665,-0.010518,0.527497,0.760462,0.0
2005-01-20,13.83,-0.6946,-0.665,-0.006109,0.536725,0.762257,0.0
2005-01-21,14.36,-0.7039,-0.667,-0.006147,0.530998,0.762539,0.0
2005-01-24,14.65,-0.7039,-0.667,-0.001972,0.529940,0.761793,0.0
2005-01-25,14.06,-0.7039,-0.667,0.002828,0.533906,0.760826,0.0


In [26]:

# Step 8: Split data chronologically

split_idx = int(len(model_df) * 0.80)

train_df = model_df.iloc[:split_idx]
test_df = model_df.iloc[split_idx:]

print("Training data:", train_df.shape)
print("Testing data:", test_df.shape)

print("\nTraining period:")
print(train_df.index.min(), "to", train_df.index.max())

print("\nTesting period:")
print(test_df.index.min(), "to", test_df.index.max())

print("\nStress labels in testing data:")
print(test_df["stress"].value_counts())


Training data: (4375, 7)
Testing data: (1094, 7)

Training period:
2005-01-19 00:00:00 to 2022-06-03 00:00:00

Testing period:
2022-06-06 00:00:00 to 2026-10-14 00:00:00

Stress labels in testing data:
stress
0.0    1004
1.0      90
Name: count, dtype: int64



### Chronological Train/Test Split

The dataset is divided into 80% training data and 20% testing data based on date order. This allows the models to be evaluated on later market observations that were not used during training.



### VIX-Only Baseline Model

Logistic regression is used to establish baseline performance using VIX as the only predictor. The results will be compared with a model that includes additional market indicators.


In [27]:

# Step 9: Train VIX-only baseline model

from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

X_train_vix = train_df[["VIXCLS"]]
X_test_vix = test_df[["VIXCLS"]]

y_train = train_df["stress"].astype(int)
y_test = test_df["stress"].astype(int)

vix_model = make_pipeline(
    StandardScaler(),
    LogisticRegression(max_iter=1000, class_weight="balanced")
)

vix_model.fit(X_train_vix, y_train)

vix_pred = vix_model.predict(X_test_vix)
vix_prob = vix_model.predict_proba(X_test_vix)[:, 1]

print("VIX-only baseline results:")
display(evaluate_model(y_test, vix_pred, vix_prob))


VIX-only baseline results:


Accuracy            0.919561
Precision           0.507463
Recall              0.755556
F1                  0.607143
ROC-AUC             0.947903
PR-AUC              0.724115
False Alarm Rate    0.065737
dtype: float64


### Combined Market Indicators Model

A logistic regression model is trained using VIX and five additional market indicators. Its performance is compared with the VIX-only baseline to evaluate whether the additional features improve stress detection.


In [28]:

# Step 10: Train combined model

X_train_combined = train_df[features]
X_test_combined = test_df[features]

combined_model = make_pipeline(
    StandardScaler(),
    LogisticRegression(max_iter=1000, class_weight="balanced")
)

combined_model.fit(X_train_combined, y_train)

combined_pred = combined_model.predict(X_test_combined)
combined_prob = combined_model.predict_proba(X_test_combined)[:, 1]

print("Combined model results:")
display(evaluate_model(y_test, combined_pred, combined_prob))


Combined model results:


Accuracy            0.887569
Precision           0.414508
Recall              0.888889
F1                  0.565371
ROC-AUC             0.957736
PR-AUC              0.750579
False Alarm Rate    0.112550
dtype: float64


### Model Performance Comparison

The VIX-only and combined models are compared using the same testing dataset and evaluation metrics. The comparison helps identify improvements in stress detection and any trade-offs in false alarms.


In [29]:

# Step 11: Compare model performance

comparison = pd.DataFrame({
    "VIX Only": evaluate_model(y_test, vix_pred, vix_prob),
    "Combined Model": evaluate_model(y_test, combined_pred, combined_prob)
})

comparison["Difference"] = (
    comparison["Combined Model"] - comparison["VIX Only"]
)

display(comparison.round(4))


,VIX Only,Combined Model,Difference
Accuracy,0.9196,0.8876,-0.0320
Precision,0.5075,0.4145,-0.0930
Recall,0.7556,0.8889,0.1333
F1,0.6071,0.5654,-0.0418
ROC-AUC,0.9479,0.9577,0.0098
PR-AUC,0.7241,0.7506,0.0265
False Alarm Rate,0.0657,0.1125,0.0468



### Results and Interpretation

The combined model achieved higher recall (88.89%) compared to the VIX-only model (75.56%), indicating that it detected more stress days.

The combined model also showed improvements in ROC-AUC and PR-AUC, suggesting better overall discrimination between stress and non-stress periods.

However, precision decreased from 50.75% to 41.55%, while the false alarm rate increased from 6.57% to 11.26%.

Overall, the additional market indicators improved stress detection but increased the number of incorrect warnings. The results show a trade-off between detecting more stress events and maintaining a lower false alarm rate.

These results reflect same-day stress detection rather than predictions of future stress.



## 5. Sentiment Feature Evaluation

FinBERT sentiment features are examined to determine whether financial text provides additional information for identifying market stress. Sentiment coverage and available dates are checked before including these features in the evaluation.


In [30]:

# Step 13: Inspect sentiment features

sentiment_path = (
    ROOT / "data" / "processed" / "features"
    / "sec_sentiment_daily.parquet"
)

sentiment_df = pd.read_parquet(sentiment_path)

print("Sentiment dataset shape:", sentiment_df.shape)
print("\nAvailable columns:")
print(sentiment_df.columns.tolist())

print("\nSample data:")
display(sentiment_df.head())


Sentiment dataset shape: (2510, 7)

Available columns:
['date', 'ticker', 'section_group', 'average_sentiment', 'tone_change', 'paragraphs', 'source_filing_trading_date']

Sample data:


,date,ticker,section_group,average_sentiment,tone_change,paragraphs,source_filing_trading_date
0,2022-01-03,AAPL,MD&A,NaN,NaN,NaN,NaT
1,2022-01-04,AAPL,MD&A,NaN,NaN,NaN,NaT
2,2022-01-05,AAPL,MD&A,NaN,NaN,NaN,NaT
3,2022-01-06,AAPL,MD&A,NaN,NaN,NaN,NaT
4,2022-01-07,AAPL,MD&A,NaN,NaN,NaN,NaT



### Sentiment Data Coverage

The availability of FinBERT sentiment scores is examined across companies and filing sections. Missing values and date coverage are checked before combining sentiment with market indicators.


In [31]:

# Step 14: Check sentiment data coverage

sentiment_df["date"] = pd.to_datetime(sentiment_df["date"])

print("Date range:")
print(sentiment_df["date"].min(), "to", sentiment_df["date"].max())

print("\nCompanies:")
print(sentiment_df["ticker"].unique())

print("\nFiling sections:")
print(sentiment_df["section_group"].unique())

print("\nMissing values:")
print(sentiment_df[[
    "average_sentiment",
    "tone_change",
    "paragraphs"
]].isna().sum())

print("\nAvailable sentiment scores:")
print(sentiment_df["average_sentiment"].notna().sum())


Date range:
2022-01-03 00:00:00 to 2022-12-30 00:00:00

Companies:
['AAPL' 'AMZN' 'GOOGL' 'MSFT' 'NVDA']

Filing sections:
['MD&A' 'Risk Factors']

Missing values:
average_sentiment     458
tone_change          1673
paragraphs            458
dtype: int64

Available sentiment scores:
2052



### Daily Sentiment Features

FinBERT sentiment scores are aggregated by trading date to create market-level sentiment indicators. Average sentiment and tone changes are used to examine whether financial text provides additional information about market stress.


In [32]:

# Step 15: Aggregate sentiment features by date

daily_sentiment = (
    sentiment_df.groupby("date")
    .agg(
        sentiment_mean=("average_sentiment", "mean"),
        tone_change_mean=("tone_change", "mean"),
        sentiment_coverage=("average_sentiment", "count")
    )
    .sort_index()
)

print("Daily sentiment shape:", daily_sentiment.shape)

print("\nMissing values:")
print(daily_sentiment.isna().sum())

print("\nSample data:")
display(daily_sentiment.head())


Daily sentiment shape: (251, 3)

Missing values:
sentiment_mean        16
tone_change_mean      79
sentiment_coverage     0
dtype: int64

Sample data:


,sentiment_mean,tone_change_mean,sentiment_coverage
date,,,
2022-01-03,NaN,NaN,0
2022-01-04,NaN,NaN,0
2022-01-05,NaN,NaN,0
2022-01-06,NaN,NaN,0
2022-01-07,NaN,NaN,0



### Sentiment and Market Data Alignment

Daily sentiment features are matched with market indicators and stress labels using trading dates. The available observations and stress distribution are checked to determine whether the sentiment data is sufficient for model evaluation.


In [33]:

# Step 16: Check sentiment and market data overlap

sentiment_model_df = model_df.join(
    daily_sentiment,
    how="inner"
)

print("Overlapping dates:", len(sentiment_model_df))

print("\nDate range:")
print(sentiment_model_df.index.min(), "to",
      sentiment_model_df.index.max())

print("\nStress label counts:")
print(sentiment_model_df["stress"].value_counts())

print("\nComplete sentiment observations:")
print(sentiment_model_df.dropna().shape[0])

print("\nSample data:")
display(sentiment_model_df.head())


Overlapping dates: 251

Date range:
2022-01-03 00:00:00 to 2022-12-30 00:00:00

Stress label counts:
stress
0.0    143
1.0    108
Name: count, dtype: int64

Complete sentiment observations:
172

Sample data:


,VIXCLS,STLFSI4,NFCI,SPY_ret,avg_corr_60d,absorption_250d,stress,sentiment_mean,tone_change_mean,sentiment_coverage
2022-01-03,16.60,-0.7601,-0.559,0.005773,0.495084,0.761477,0.0,NaN,NaN,0
2022-01-04,16.91,-0.7601,-0.559,-0.000335,0.480629,0.763289,0.0,NaN,NaN,0
2022-01-05,19.73,-0.7601,-0.559,-0.019389,0.480661,0.763981,0.0,NaN,NaN,0
2022-01-06,19.61,-0.7601,-0.559,-0.000940,0.472558,0.763707,0.0,NaN,NaN,0
2022-01-07,18.76,-0.7009,-0.564,-0.003961,0.469405,0.762950,0.0,NaN,NaN,0



### Sentiment Model Data Preparation

Observations with complete sentiment and market indicators are selected for evaluation. The remaining stress distribution is checked to assess whether the available data supports a meaningful comparison.


In [34]:

# Step 17: Prepare complete sentiment observations

sentiment_complete = sentiment_model_df.dropna().copy()

print("Complete observations:", len(sentiment_complete))

print("\nStress label distribution:")
print(sentiment_complete["stress"].value_counts())

print("\nDate range:")
print(sentiment_complete.index.min(), "to",
      sentiment_complete.index.max())

print("\nStress percentage:")
print(round(sentiment_complete["stress"].mean() * 100, 2), "%")


Complete observations: 172

Stress label distribution:
stress
1.0    92
0.0    80
Name: count, dtype: int64

Date range:
2022-04-27 00:00:00 to 2022-12-30 00:00:00

Stress percentage:
53.49 %



### Sentiment Model Comparison

A separate comparison is prepared using the available 2022 sentiment data. Market indicators are evaluated with and without FinBERT sentiment features to examine their potential contribution to stress detection.

The results are treated as exploratory because the sentiment dataset covers only five companies and one year.


In [35]:

# Step 18: Prepare sentiment model comparison

market_features = features

sentiment_features = [
    "sentiment_mean",
    "tone_change_mean",
    "sentiment_coverage"
]

all_features = market_features + sentiment_features

split_sentiment = int(len(sentiment_complete) * 0.80)

sentiment_train = sentiment_complete.iloc[:split_sentiment]
sentiment_test = sentiment_complete.iloc[split_sentiment:]

print("Training observations:", len(sentiment_train))
print("Testing observations:", len(sentiment_test))

print("\nTraining stress labels:")
print(sentiment_train["stress"].value_counts())

print("\nTesting stress labels:")
print(sentiment_test["stress"].value_counts())

print("\nTesting period:")
print(sentiment_test.index.min(), "to", sentiment_test.index.max())


Training observations: 137
Testing observations: 35

Training stress labels:
stress
1.0    81
0.0    56
Name: count, dtype: int64

Testing stress labels:
stress
0.0    24
1.0    11
Name: count, dtype: int64

Testing period:
2022-11-10 00:00:00 to 2022-12-30 00:00:00



### Evaluating the Contribution of Sentiment

Two logistic regression models are trained using the same 2022 observations. The first uses market indicators only, while the second includes FinBERT sentiment features.

Their performance is compared to examine whether sentiment provides additional information for identifying market stress.


In [36]:

# Step 19: Compare models with and without sentiment

y_train_sent = sentiment_train["stress"].astype(int)
y_test_sent = sentiment_test["stress"].astype(int)

market_only_model = make_pipeline(
    StandardScaler(),
    LogisticRegression(max_iter=1000, class_weight="balanced")
)

market_only_model.fit(
    sentiment_train[market_features], y_train_sent
)

market_only_pred = market_only_model.predict(
    sentiment_test[market_features]
)
market_only_prob = market_only_model.predict_proba(
    sentiment_test[market_features]
)[:, 1]

with_sentiment_model = make_pipeline(
    StandardScaler(),
    LogisticRegression(max_iter=1000, class_weight="balanced")
)

with_sentiment_model.fit(
    sentiment_train[all_features], y_train_sent
)

with_sentiment_pred = with_sentiment_model.predict(
    sentiment_test[all_features]
)
with_sentiment_prob = with_sentiment_model.predict_proba(
    sentiment_test[all_features]
)[:, 1]

sentiment_comparison = pd.DataFrame({
    "Market Only": evaluate_model(
        y_test_sent, market_only_pred, market_only_prob
    ),
    "Market + Sentiment": evaluate_model(
        y_test_sent, with_sentiment_pred, with_sentiment_prob
    )
})

sentiment_comparison["Difference"] = (
    sentiment_comparison["Market + Sentiment"]
    - sentiment_comparison["Market Only"]
)

display(sentiment_comparison.round(4))


,Market Only,Market + Sentiment,Difference
Accuracy,0.6857,0.7429,0.0571
Precision,0.0000,1.0000,1.0000
Recall,0.0000,0.1818,0.1818
F1,0.0000,0.3077,0.3077
ROC-AUC,0.8864,0.8977,0.0114
PR-AUC,0.8400,0.8491,0.0091
False Alarm Rate,0.0000,0.0000,0.0000



### Sentiment Evaluation Results

Adding FinBERT sentiment features increased accuracy from 68.57% to 74.29% and improved ROC-AUC from 88.64% to 89.77%.

The market-only model did not identify any stress days at the default classification threshold. With sentiment features, the model correctly identified 2 out of 11 stress days without generating false alarms.

Although sentiment improved several evaluation metrics, recall remained low at 18.18%.

These findings suggest that sentiment features may provide additional information for market stress detection. However, the results are exploratory because the evaluation uses only 35 testing observations from 2022 and sentiment data from five companies.



### Stress Label Methodology

Historical stress labels are based on a centered 21-day rolling volatility measure of the S&P 500, using a 25% annualized volatility threshold.

Because the rolling window is centered, the labels include information from future trading days. These labels are useful for identifying historical stress periods but are not available in real time.

The current evaluation measures how well market indicators identify historically labeled stress days. It does not establish whether the models can predict stress before it occurs.

A future early-warning evaluation would require a clearly defined prediction horizon and features available at the time of prediction.



## 6. Conclusion and Limitations

### Key Findings

The VIX-only and combined market models were evaluated using historical stress labels and a chronological train/test split.

The combined model achieved higher recall (88.89%) than the VIX-only model (75.56%), showing improved stress detection. However, the false alarm rate also increased from 6.57% to 11.26%.

Adding FinBERT sentiment features improved several metrics in the separate 2022 evaluation. However, the sentiment model identified only 2 out of 11 stress days in the testing period.

### Limitations

- Historical stress labels are calculated using centered rolling volatility, which includes future observations. The current models evaluate same-day stress detection rather than real-time early-warning performance.
- Sentiment data is limited to five companies during 2022.
- The sentiment evaluation includes only 35 testing observations.
- Market conditions may change over time, affecting model performance.
- Additional historical data and testing are needed to confirm whether sentiment consistently improves stress detection.

### Next Steps

Future work should evaluate predictions of upcoming stress events, expand sentiment coverage, and test model performance across different market conditions.
